# Dataset Benchmark: Open Bandit Dataset (OBD)

Основной эксперимент на реальных логах [Open Bandit Dataset](https://github.com/st-tech/zr-obp/tree/master/obd) (ZOZOTOWN, fashion e-commerce).

Перед запуском необходимо подготовить данные:

```bash
python -m scripts.prepare_open_bandit --download --behavior-policy random --campaign all --include-context --output-path data/processed/obd_events.csv
```

Ниже сравниваются `fixed_ab`, `epsilon_greedy`, `ucb1`, `thompson_sampling` в режиме `batch` (simulator-like) и `ope` (честная offline-оценка с propensity).

In [3]:
from pathlib import Path

from src.bandits.epsilon_greedy import EpsilonGreedyPolicy
from src.bandits.fixed_ab import FixedABPolicy
from src.bandits.thompson_sampling import ThompsonSamplingPolicy
from src.bandits.ucb1 import UCB1Policy
from src.environments.logged_clicks import LoggedClicksBanditEnv
from src.experiments.configs import ExperimentConfig
from src.experiments.runner import compare_policies
from src.evaluation.summary import summarize_runs
from src.ope.replay import compare_policies_replay
from src.pipeline.loader import load_events

EVENTS_PATH = Path("../data/processed/obd_events.csv")
if not EVENTS_PATH.exists():
    raise FileNotFoundError(
        "Сначала подготовьте OBD: python -m scripts.prepare_open_bandit --download --include-context"
    )

events = load_events(EVENTS_PATH)
policies = {
    "fixed_ab": lambda n_arms, seed: FixedABPolicy(n_arms=n_arms, seed=seed),
    "epsilon_greedy": lambda n_arms, seed: EpsilonGreedyPolicy(n_arms=n_arms, epsilon=0.1, seed=seed),
    "ucb1": lambda n_arms, seed: UCB1Policy(n_arms=n_arms, seed=seed),
    "thompson_sampling": lambda n_arms, seed: ThompsonSamplingPolicy(n_arms=n_arms, seed=seed),
}

batch_configs = []
for seed in range(5):
    for name, factory in policies.items():
        batch_configs.append(
            ExperimentConfig(
                run_id=f"{name}_seed{seed}",
                seed=seed,
                horizon=len(events),
                mode="batch",
                batch_size=200,
                policy_factory=factory,
                environment_factory=lambda s, loaded=events: LoggedClicksBanditEnv(events=loaded, seed=s),
            )
        )

batch_results = compare_policies(batch_configs)
batch_summary = summarize_runs(batch_results)
print("Batch mode (simulator-like):")
display(
    batch_summary.groupby("policy_name")[
        ["final_cumulative_reward", "final_cumulative_regret", "suboptimal_share"]
    ].mean().sort_values("final_cumulative_regret")
)

ope_summary = compare_policies_replay(events=events, policy_factories=policies, seeds=5)
print("Strict OPE mode (IPS/SNIPS with propensity):")
display(
    ope_summary.groupby("policy_name")[
        ["acceptance_rate", "ips_estimate", "snips_estimate", "effective_sample_size"]
    ].mean().sort_values("snips_estimate", ascending=False)
)

Batch mode (simulator-like):


,final_cumulative_reward,final_cumulative_regret,suboptimal_share
policy_name,,,
epsilon_greedy,175.0,169.827586,0.86138
thompson_sampling,128.0,216.827586,0.97458
ucb1,114.8,230.027586,0.98000
fixed_ab,109.0,235.827586,0.98634


Strict OPE mode (IPS/SNIPS with propensity):


,acceptance_rate,ips_estimate,snips_estimate,effective_sample_size
policy_name,,,,
fixed_ab,0.01314,0.0032,0.003078,131.4
thompson_sampling,0.01184,0.0016,0.001754,118.4
epsilon_greedy,0.01260,0.0000,0.000000,126.0
ucb1,0.00990,0.0000,0.000000,99.0


In [ ]:
from IPython.display import Image, display
from pathlib import Path

fig_dir = Path("../outputs/figures")
for name in sorted(fig_dir.glob("fig_3_*.png")):
    print(name.name)
    display(Image(filename=str(name), width=700))